In [4]:
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torchvision.ops import nms, roi_align,box_iou
from collections import defaultdict
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as TF
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

dataloader

In [5]:
# VOC 官方的 20 个前景类别；0 保留给背景。
VOC_CLASSES = (
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
)
CLASS_TO_INDEX = {name: index + 1 for index, name in enumerate(VOC_CLASSES)}
#把 torchvision 的 VOC XML 标注转换为 image、boxes、labels
class VOCDataset(Dataset):
    """把 torchvision 的 VOC XML 标注转换为 image、boxes、labels。"""

    def __init__(
        self,
        root: str,
        image_set: str = "trainval",
        image_size: tuple[int, int] = (448, 448),
        train: bool = True,
        download: bool = False,
    ) -> None:
        self.dataset = VOCDetection(
            root=root,
            year="2007",
            image_set=image_set,
            download=download,
        )
        self.image_size = image_size  # (目标高度, 目标宽度)
        self.train = train

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor]:
        image, target = self.dataset[index]
        image = image.convert("RGB")
        original_width, original_height = image.size

        annotation = target["annotation"]
        objects = annotation.get("object", [])
        if isinstance(objects, dict):
            objects = [objects]

        boxes = []
        labels = []
        for obj in objects:
            box = obj["bndbox"]

            # VOC XML 坐标从 1 开始，这里转成从 0 开始的 xyxy。
            xmin = float(box["xmin"]) - 1
            ymin = float(box["ymin"]) - 1
            xmax = float(box["xmax"]) - 1
            ymax = float(box["ymax"]) - 1
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_TO_INDEX[obj["name"]])

        if not boxes:
            raise ValueError(f"VOC 样本 {index} 没有目标框")

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        labels_tensor = torch.tensor(labels, dtype=torch.long)

        # 当前模型用 torch.stack 组成 batch，所以所有图片统一为固定大小。
        target_height, target_width = self.image_size
        scale_x = target_width / original_width
        scale_y = target_height / original_height
        boxes_tensor[:, [0, 2]] *= scale_x
        boxes_tensor[:, [1, 3]] *= scale_y
        #这里的resize是进行拉伸
        image = TF.resize(image, [target_height, target_width])

        # 水平翻转图像时，边界框也必须一起翻转。
        if self.train and torch.rand(()) < 0.5:
            #进行水平翻转
            image = TF.hflip(image)
            old_xmin = boxes_tensor[:, 0].clone()
            old_xmax = boxes_tensor[:, 2].clone()
            boxes_tensor[:, 0] = target_width - old_xmax
            boxes_tensor[:, 2] = target_width - old_xmin

        image_tensor = TF.to_tensor(image)  # [0, 255] -> [0, 1]
        image_tensor = TF.normalize(
            image_tensor,
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        )
        return image_tensor, boxes_tensor, labels_tensor
#返回images，boxes，labels，images为tensor形式，boxes和labels为list形式
def detection_collate(batch):
    """图片尺寸一致可以 stack；每张图框数不同，所以 boxes/labels 保留 list。"""
    images, boxes, labels = zip(*batch)
    return torch.stack(images), list(boxes), list(labels)
def build_voc2007_dataloader(
    root: str,
    batch_size: int = 2,
    image_size: tuple[int, int] = (448, 448),
    image_set: str = "trainval",
    train: bool = True,
    download: bool = True,
    num_workers: int = 0,
) -> DataLoader:
    dataset = VOCDataset(
        root=root,
        image_set=image_set,
        image_size=image_size,
        train=train,
        download=download,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=detection_collate,
    )

model

In [6]:
#把网络预测出的框偏移量应用到原始框上，恢复出真正的预测框，解码操作
def decode_boxes(deltas: Tensor, boxes: Tensor) -> Tensor:
    """把 (dx, dy, dw, dh) 还原成 (x1, y1, x2, y2)。
        这里deltas指的是回归头预测的t,维度[预测框数量,4(x1,y1,x2,y2)]
        boxes指的是预测框,维度[预测框数量,4(x1,y1,x2,y2)]
    """
    
    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    #计算中心点
    ctr_x = boxes[:, 0] + 0.5 * widths
    ctr_y = boxes[:, 1] + 0.5 * heights

    dx, dy, dw, dh = deltas.unbind(dim=1)
    pred_ctr_x = dx * widths + ctr_x
    pred_ctr_y = dy * heights + ctr_y
    #限制高宽的倍率防止数值爆炸
    pred_w = dw.clamp(max=math.log(1000 / 16)).exp() * widths
    pred_h = dh.clamp(max=math.log(1000 / 16)).exp() * heights

    #返回解码框的[x1,y1,x2,y2]
    return torch.stack(
        (
            pred_ctr_x - 0.5 * pred_w,
            pred_ctr_y - 0.5 * pred_h,
            pred_ctr_x + 0.5 * pred_w,
            pred_ctr_y + 0.5 * pred_h,
        ),
        dim=1,
    )
#限制（x1，y1，x2，y2）在image_size里，不超出图片
def clip_boxes(boxes: Tensor, image_size: tuple[int, int]) -> Tensor:
    """把框限制在图像内；image_size 为 (H, W)。"""
    h, w = image_size
    x1 = boxes[:, 0].clamp(0, w)
    y1 = boxes[:, 1].clamp(0, h)
    x2 = boxes[:, 2].clamp(0, w)
    y2 = boxes[:, 3].clamp(0, h)
    return torch.stack((x1, y1, x2, y2), dim=1)
#Backbone结构，提取图片特征，返回[B,C,H,W]
class TinyBackbone(nn.Module):
    """VGG 风格主干；4 次池化，所以特征图步长为 16。"""

    out_channels = 512
    #经过4次池化，VGG输出图像的每一个像素点相当于原图片的16个像素点
    stride = 16

    def __init__(self) -> None:
        super().__init__()

        def block(in_channels: int, out_channels: int, n_conv: int) -> list[nn.Module]:
            layers: list[nn.Module] = []
            for _ in range(n_conv):
                layers += [nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU()]
                in_channels = out_channels
            layers.append(nn.MaxPool2d(2))
            return layers

        self.body = nn.Sequential(
            *block(3, 64, 2),
            *block(64, 128, 2),
            *block(128, 256, 3),
            *block(256, 512, 3),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(),
        )

    def forward(self, images: Tensor) -> Tensor:
        return self.body(images)
#为featuremap每个像素点预测框，最终输出[像素点框的集合，框的坐标]
#返回的为在原图片中的坐标
class AnchorGenerator(nn.Module):
    """在特征图每个位置生成 3 个比例 x 3 个尺度 = 9 个 anchors。"""

    def __init__(
        self,
        stride: int = 16,
        scales: tuple[int, ...] = (4, 8, 16),
        ratios: tuple[float, ...] = (0.5, 1.0, 2.0),
    ) -> None:
        super().__init__()
        #stride表示feature map上一个点对应原图一条边的长度
        self.stride = stride
        #scale表示anchor在featuremap上的大小，如果是4则代表边长是4
        self.scales = scales
        #ratio表示高宽比例(w/h)
        self.ratios = ratios

    #表示可以以类的属性形式返回
    #num_anchors表示按照不同的比例和scale倍数生成不同数量的框
    @property
    def num_anchors(self) -> int:
        return len(self.scales) * len(self.ratios)

    def forward(self, feature: Tensor) -> Tensor:
        #backbone输出[batchsize,num_channels,h,w]
        _, _, h, w = feature.shape
        dtype, device = feature.dtype, feature.device

        base_anchors = []
        for ratio in self.ratios:
            for scale in self.scales:
                area_side = self.stride * scale
                anchor_w = area_side / math.sqrt(ratio)
                anchor_h = area_side * math.sqrt(ratio)
                #base_anchors的维度是[num_anchors,anchorsize]
                base_anchors.append([-anchor_w / 2, -anchor_h / 2,
                                     anchor_w / 2, anchor_h / 2])
        base = torch.tensor(base_anchors, dtype=dtype, device=device)

        shifts_x = (torch.arange(w, dtype=dtype, device=device) + 0.5) * self.stride
        shifts_y = (torch.arange(h, dtype=dtype, device=device) + 0.5) * self.stride
        #这里返回的grid_x和grid_y的形状都是[h,w]，表示转换到原图片的中心网格点
        grid_y, grid_x = torch.meshgrid(shifts_y, shifts_x, indexing="ij")

        #shifts的维度是[H*W,4]
        shifts = torch.stack((grid_x, grid_y, grid_x, grid_y), dim=-1).reshape(-1, 4)
        #最终return的维度是[H*W,num_anchors,anchorsize]
        return (shifts[:, None, :] + base[None, :, :]).reshape(-1, 4)
#把 RPN 对所有 anchor 的预测结果，整理成一小批质量较高的 proposal 候选框，交给后面的 RoI Head。
#返回经score筛选后的解码后anchor
#返回boxes[框的数量,4]
class ProposalGenerator(nn.Module):
    """RPN 后处理：解码、裁剪、小框过滤、Top-K、NMS。"""

    def __init__(
        self,
        pre_nms_topk: int = 6000,
        #表示在做 NMS 之前，最多先留下 objectness 分数最高的 6000 个框。
        post_nms_topk: int = 300,
        #表示做完 NMS 之后，最终最多保留 300 个 proposal。
        nms_thresh: float = 0.7,
        #表示如果两个候选框 IoU 很高，达到 NMS 的抑制条件，就把重复框删掉。这里阈值是 0.7
        min_size: float = 8.0,
        #表示宽或高小于 8 个像素的框直接删除
    ) -> None:
        super().__init__()
        self.pre_nms_topk = pre_nms_topk
        self.post_nms_topk = post_nms_topk
        self.nms_thresh = nms_thresh
        self.min_size = min_size

    def forward(
        self, anchors: Tensor, 
              #原始anchor位置，维度[框的数量，4]
              deltas: Tensor,
              #RPN回归头预测的框偏移量,维度[框的数量,4]
              scores: Tensor,
              #RPN预测的前景/objectness分数,分类头预测结果，维度[框的数量]
              image_size: tuple[int, int],
              #表示图片的高宽
    ) -> Tensor:
        #将框转变为解码后的预测框
        boxes = clip_boxes(decode_boxes(deltas, anchors), image_size)
        widths = boxes[:, 2] - boxes[:, 0]
        heights = boxes[:, 3] - boxes[:, 1]
        #保证高和宽都大于最小值
        keep = (widths >= self.min_size) & (heights >= self.min_size)
        boxes, scores = boxes[keep], scores[keep]

        #在scores中仅保留分数前pre_nms_topk个数据，返回的是索引
        order = scores.argsort(descending=True)[: self.pre_nms_topk]
        #仅保留boxes和scores中前pre_num_topk个框
        boxes, scores = boxes[order], scores[order]
        #重复进行NMS最终仅保留post_nums_topk个框,返回索引
        keep = nms(boxes, scores, self.nms_thresh)[: self.post_nms_topk]
        return boxes[keep]
#提供候选框给ROI head
#返回proposal维度[B,H*W*A,4],B维是list格式
#           计算loss用{
          #  "objectness": objectness,分类头结果[B,H*W*A,2]
           # "box_deltas": deltas,回归头结果[B,H*W*A,4]
           # "anchors": anchors,对每个像素点生成的框[H*W*A,anchor_size]
      #  }
class RPN(nn.Module):
    """Region Proposal Network：对每个 anchor 判断物体性并回归位置。"""

    def __init__(self, in_channels: int, anchor_generator: AnchorGenerator) -> None:
        super().__init__()
        '''这里inchannel指的是backbone输出的channel数
            这里的anchor_generator是预测每个像素点的anchor
        '''
        self.anchor_generator = anchor_generator
        #获取RPN筛选出的proposal anchor
        self.proposal_generator = ProposalGenerator()
        #获取featuremap上一个像素点预测anchor的数量
        a = anchor_generator.num_anchors
        #RPN的共享特征提取层
        self.conv = nn.Conv2d(in_channels, in_channels, 3, padding=1)
        #分类头，二分类（backgroud/foreground）
        self.objectness = nn.Conv2d(in_channels, a * 2, 1)
        #回归头，预测delta(t)
        self.box_deltas = nn.Conv2d(in_channels, a * 4, 1)

    def forward(
        self, feature: Tensor, image_size: tuple[int, int],
    ) -> tuple[list[Tensor], dict[str, Tensor]]:
        #共享特征提取层
        #X的维度[B,C,H,W]
        x = F.relu(self.conv(feature))
        b, _, h, w = x.shape
        a = self.anchor_generator.num_anchors

        # [B, A*2, H, W] -> [B, A, 2,H,W]
        objectness = self.objectness(x).view(b, a, 2, h, w)
        #[B, A, 2,H,W]->[B,H,W,A,2]->[b,H*W*A,2],分类头预测结果
        objectness = objectness.permute(0, 3, 4, 1, 2).reshape(b, -1, 2)
        #[B,A*4,H,W]->[B,A,4,H,W]->[B,H*W*A,4],回归头预测结果
        deltas = self.box_deltas(x).view(b, a, 4, h, w)
        deltas = deltas.permute(0, 3, 4, 1, 2).reshape(b, -1, 4)
        #返回anchor，[H*W,anchors,anchorsize]
        anchors = self.anchor_generator(feature)
        #[...,1]表示前面的维度都保留，最后一个维度仅保留索引1
        foreground_scores = objectness.softmax(dim=-1)[..., 1]
        #生成每个batch的proposal anchor[B,]
        proposals = [
            self.proposal_generator(
                anchors, deltas[i].detach(), foreground_scores[i].detach(), image_size
            )
            for i in range(b)
        ]
        
        return proposals, {
            "objectness": objectness,
            "box_deltas": deltas,
            "anchors": anchors,
        }
#输出ROI回归头和分类头的输出[B*N,num_classes],[B*N,num_classes,4]
#每个proposal对每个classes都会进行预测
class RoIHead(nn.Module):
    """对每个 proposal 做 RoIAlign，再进行分类和类别相关的框回归。"""

    def __init__(self, in_channels: int, num_classes: int, spatial_scale: float) -> None:
        '''in_channels表示Backbone输出特征图的通道数
            num_classes表示类别数量，包含背景类
            spatital表示放到featuremap要缩小多少倍，stride的倒数
        '''
        super().__init__()
        self.spatial_scale = spatial_scale
        self.num_classes = num_classes
        #ROI_align将返回[proposal总数，C,7,7]
        self.fc = nn.Sequential(
            nn.Linear(in_channels * 7 * 7, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
        )
        #分类器输出logits
        self.classifier = nn.Linear(1024, num_classes)
        #回归头输出delta
        self.box_regressor = nn.Linear(1024, num_classes * 4)

    def forward(self, feature: Tensor, proposals: list[Tensor]) -> tuple[Tensor, Tensor]:
        pooled = roi_align(
            feature,#[B,C,H,W]
            proposals,#[B,N,4]
            output_size=(7, 7),
            spatial_scale=self.spatial_scale,
            sampling_ratio=2,#采样数
            aligned=True,#做更精确的坐标对正
        )
        #从第一维开始，把后边的数据都flattern
        x = self.fc(pooled.flatten(1))
        class_logits = self.classifier(x)
        box_deltas = self.box_regressor(x).reshape(-1, self.num_classes, 4)
        return class_logits, box_deltas
#完整循环，最后返回2个回归头和分类头的值
#返回proposal维度[B,N,4],B维是list格式
# "objectness": objectness,分类头结果[B,H*W*A,2]
# "box_deltas": deltas,回归头结果[B,H*W*A,4]
# "anchors": anchors,对每个像素点生成的框[H*W*A,anchor_size]
# "roi_class_logits": class_logits,[N_total,num_classes]
# "roi_box_deltas": roi_box_deltas,[N_total,num_classes,4]
class FasterRCNN(nn.Module):
    """Faster R-CNN 总装类；num_classes 包含背景类 0。"""

    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.backbone = TinyBackbone()
        anchors = AnchorGenerator(stride=self.backbone.stride)
        self.rpn = RPN(self.backbone.out_channels, anchors)
        self.roi_head = RoIHead(
            self.backbone.out_channels,
            num_classes,
            spatial_scale=1.0 / self.backbone.stride,
        )

    def forward(self, images: Tensor) -> dict[str, Tensor | list[Tensor]]:
        """images: [B, 3, H, W]，同一批图像需具有相同大小。"""
        #获得image_size[H,W]
        image_size = (images.shape[-2], images.shape[-1])
        #输出[B,C,H,W]
        feature = self.backbone(images)
        #输出proposal和rpn计算loss
        proposals, rpn_output = self.rpn(feature, image_size)
        class_logits, roi_box_deltas = self.roi_head(feature, proposals)

        return {
            "proposals": proposals,
            #[B,N,4],B维度是list
            "roi_class_logits": class_logits,
            #[B*N,num_classes]
            "roi_box_deltas": roi_box_deltas,
            #[B*N,num_classes*4]
            "rpn_objectness": rpn_output["objectness"],
            #[B,H*W*A,2]
            "rpn_box_deltas": rpn_output["box_deltas"],
            #[B,H*W*A,4]
            "anchors": rpn_output["anchors"],
            #[H*W*A,4]
        }

train

In [7]:
#把 box -> gt_box 的变化编码为 (dx, dy, dw, dh)
def encode_boxes(boxes: Tensor, gt_boxes: Tensor) -> Tensor:
    """把 box -> gt_box 的变化编码为 (dx, dy, dw, dh)。"""
    eps = torch.finfo(boxes.dtype).eps
    widths = (boxes[:, 2] - boxes[:, 0]).clamp(min=eps)
    heights = (boxes[:, 3] - boxes[:, 1]).clamp(min=eps)
    ctr_x = boxes[:, 0] + 0.5 * widths
    ctr_y = boxes[:, 1] + 0.5 * heights

    gt_widths = (gt_boxes[:, 2] - gt_boxes[:, 0]).clamp(min=eps)
    gt_heights = (gt_boxes[:, 3] - gt_boxes[:, 1]).clamp(min=eps)
    gt_ctr_x = gt_boxes[:, 0] + 0.5 * gt_widths
    gt_ctr_y = gt_boxes[:, 1] + 0.5 * gt_heights

    dx = (gt_ctr_x - ctr_x) / widths
    dy = (gt_ctr_y - ctr_y) / heights
    dw = torch.log(gt_widths / widths)
    dh = torch.log(gt_heights / heights)
    return torch.stack((dx, dy, dw, dh), dim=1)
#在rpn计算时提供随机样本
def subsample(labels: Tensor, total: int, positive_fraction: float) -> Tensor:
    """随机保留一部分正负样本，其余设为 -1（忽略）。"""
    labels = labels.clone()
    positive = torch.where(labels == 1)[0]
    negative = torch.where(labels == 0)[0]

    num_positive = min(int(total * positive_fraction), positive.numel())
    num_negative = min(total - num_positive, negative.numel())

    disable_positive = positive[torch.randperm(positive.numel(), device=labels.device)[num_positive:]]
    disable_negative = negative[torch.randperm(negative.numel(), device=labels.device)[num_negative:]]
    labels[disable_positive] = -1
    labels[disable_negative] = -1
    return labels
#根据iou值给每个样本分配0，1，-1的标签，在求分类头损失时不考虑-1标签的框
#  在求回归头损失时只考虑1标签
def make_rpn_targets(
    anchors: Tensor,
    gt_boxes: Tensor,
    image_size: tuple[int, int],
) -> tuple[Tensor, Tensor]:
    """给 anchors 分配正样本、负样本和回归目标。"""
    h, w = image_size
    #返回所有在图中的框[H*W*A]
    inside = (
        (anchors[:, 0] >= 0)
        & (anchors[:, 1] >= 0)
        & (anchors[:, 2] <= w)
        & (anchors[:, 3] <= h)
    )

    #获得所有在图片内框的索引
    inside_index = torch.where(inside)[0]
    inside_anchors = anchors[inside_index]

    #计算在图片内的框和目标框的iou[N,M]
    ious = box_iou(inside_anchors, gt_boxes)
    #获得每个框的最大iou值和对应的索引
    max_iou, matched_gt = ious.max(dim=1)

    # 1=正样本，0=负样本，-1=忽略
    inside_labels = torch.full(
        (inside_anchors.shape[0],), -1, dtype=torch.long, device=anchors.device
    )
    inside_labels[max_iou < 0.3] = 0
    inside_labels[max_iou >= 0.7] = 1

    # 保证每个真实框至少有一个正 anchor。
    #即便有tg_box的iou值没达到0.7也有正样本
    #返回每个tg_box对应的最大iou值
    best_iou_per_gt = ious.max(dim=0).values
    best_for_any_gt = (ious == best_iou_per_gt[None]).any(dim=1)
    inside_labels[best_for_any_gt] = 1

    #抽取256个样本，正样本占比0.5
    inside_labels = subsample(inside_labels, total=256, positive_fraction=0.5)

    labels = torch.full(
        (anchors.shape[0],), -1, dtype=torch.long, device=anchors.device
    )
    labels[inside_index] = inside_labels

    box_targets = torch.zeros_like(anchors)
    box_targets[inside_index] = encode_boxes(
        inside_anchors, gt_boxes[matched_gt]
    )
    return labels, box_targets
#为ROI head返回采样预测框，类别和回归目标
def make_roi_targets(
    proposals: Tensor,#维度为[N,4]
    gt_boxes: Tensor,#维度为[M,4]
    gt_labels: Tensor,#[M]
) -> tuple[Tensor, Tensor, Tensor]:
    """从 proposals 中抽取 128 个 RoI，并分配类别及回归目标。"""
    #将目标框与预测框concat，使得模型获得高质量样本
    proposals = torch.cat((proposals.detach(), gt_boxes), dim=0)
    #计算每一个proposal和真实GT之间的iou
    #  得到一个[N,M]的IOU矩阵
    ious = box_iou(proposals, gt_boxes)
    #返回每个proposal对应的标号，指定维度的.max不仅会返回数值，还会返回索引
    #  matched_gt代表配对的gt_boxes的索引
    max_iou, matched_gt = ious.max(dim=1)

    #获取max_iou>=0.5的索引，正样本
    positive = torch.where(max_iou >= 0.5)[0]
    #获取max_iou<0.5的索引，负样本
    negative = torch.where(max_iou < 0.5)[0]
    #正样本最多采32个
    num_positive = min(32, positive.numel()) 
    #负样本最多采96个
    num_negative = min(128 - num_positive, negative.numel())

    #随机抽取positive和negative的样本
    positive = positive[torch.randperm(positive.numel(), device=proposals.device)[:num_positive]]
    negative = negative[torch.randperm(negative.numel(), device=proposals.device)[:num_negative]]
    #这里keep为在proposals中的索引
    keep = torch.cat((positive, negative))

    #抽取的候选框
    sampled_proposals = proposals[keep]
    matched_gt = matched_gt[keep]
    #获得类型标号
    labels = gt_labels[matched_gt].clone()
    #把负样本的标号变成0
    labels[num_positive:] = 0
    #获取真实t
    box_targets = encode_boxes(sampled_proposals, gt_boxes[matched_gt])
    return sampled_proposals, labels, box_targets
#计算ROIhead的回归头损失
def box_regression_loss(
    predicted: Tensor,
    target: Tensor,
    positive: Tensor,
    normalizer: int,
) -> Tensor:
    """只对正样本计算 Smooth L1；再用总采样数归一化。"""
    if not positive.any():
        return predicted.sum() * 0.0
    #计算平均smoothl1loss
    return F.smooth_l1_loss(
        predicted[positive], target[positive], beta=1 / 9, reduction="sum"
    ) / max(normalizer, 1)
#返回losses，total为过程中总损失
def compute_losses(
    model: FasterRCNN,
    images: Tensor,
    gt_boxes: list[Tensor],
    gt_labels: list[Tensor],
) -> dict[str, Tensor]:
    """执行训练前向，并返回 Faster R-CNN 的四项 loss。"""
    #获取image的H和W
    image_size = (images.shape[-2], images.shape[-1])
    #经过backbone获得featuremap
    feature = model.backbone(images)
    #获得预测框和rpn的输出
    proposals, rpn_output = model.rpn(feature, image_size)

    rpn_cls_losses = []
    rpn_box_losses = []
    sampled_proposals = []
    roi_labels = []
    roi_box_targets = []

    #这里是对batchsize中每个图片进行处理
    for i, (boxes, labels) in enumerate(zip(gt_boxes, gt_labels)):
        #anchor_labels返回样本分类标签形状[H*W*A]
        #  这里通过与真实框的IOU给予label标签，分为1，0，-1
        #anchor_box_target表示要达到真实的边界框的delta(t),[H*W*A,4]
        anchor_labels, anchor_box_targets = make_rpn_targets(
            rpn_output["anchors"], boxes, image_size
        )
        #计算loss忽略IOU小的label数据[-1]
        rpn_cls_losses.append(
            F.cross_entropy(
                rpn_output["objectness"][i], anchor_labels, ignore_index=-1
            )
        )
        #计算回归头的loss时仅计算label为1的loss
        rpn_box_losses.append(
            box_regression_loss(
                rpn_output["box_deltas"][i],
                anchor_box_targets,
                anchor_labels == 1,
                normalizer=(anchor_labels >= 0).sum().item(),
            )
        )
        #sampled是从RPN产生的proposal中采样出的候选框[k,4]
        #labels 是这些 sampled proposals 对应的分类标签[K]
        #targets 是这些 sampled proposals 对应的边框回归目标[K,4]
        sampled, labels, targets = make_roi_targets(
            proposals[i], boxes, labels
        )
        sampled_proposals.append(sampled)
        roi_labels.append(labels)
        roi_box_targets.append(targets)

    #将batchsize集合，输出是ROI回归头和分类头的logits
    roi_class_logits, roi_box_deltas = model.roi_head(
        feature, sampled_proposals
    )
    roi_labels_tensor = torch.cat(roi_labels)
    roi_box_targets_tensor = torch.cat(roi_box_targets)

    #计算ROI分类头loss
    roi_cls_loss = F.cross_entropy(roi_class_logits, roi_labels_tensor)

    # RoI Head 会为每个类别都预测 4 个数，只取真实类别对应的那一组。
    row = torch.arange(roi_labels_tensor.numel(), device=images.device)
    #获取anchor对应的类型的预测anchor
    selected_deltas = roi_box_deltas[row, roi_labels_tensor]
    roi_box_loss = box_regression_loss(
        selected_deltas,
        roi_box_targets_tensor,
        #保留roi_labels_tensor>0的样本
        roi_labels_tensor > 0,
        #最后要除的样本数
        normalizer=roi_labels_tensor.numel(),
    )

    losses = {
        "rpn_cls": torch.stack(rpn_cls_losses).mean(),
        "rpn_box": torch.stack(rpn_box_losses).mean(),
        "roi_cls": roi_cls_loss,
        "roi_box": roi_box_loss,
    }
    losses["total"] = sum(losses.values())
    return losses
#训练一轮循环，最后返回loss数据
def train_one_epoch(
    model: FasterRCNN,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> dict[str, float]:
    model.train()
    totals: defaultdict[str, float] = defaultdict(float)

    for images, boxes, labels in loader:
        #把数据转入device
        images = images.to(device)
        boxes = [box.to(device) for box in boxes]
        labels = [label.to(device) for label in labels]

        losses = compute_losses(model, images, boxes, labels)
        optimizer.zero_grad()
        losses["total"].backward()
        optimizer.step()

        for name, value in losses.items():
            totals[name] += value.item()

    return {name: value / len(loader) for name, value in totals.items()}
#返回images，boxes，labels，images为tensor形式，boxes和labels为list形式
#训练综合循环
def traintotal(epochs,num_classes,dataloder,writer) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FasterRCNN(num_classes=num_classes).to(device)  # 背景 + Toy 的两个类别

    optimizer = torch.optim.SGD(
        model.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4
    )

    for epoch in range(epochs):
        losses = train_one_epoch(model, dataloder, optimizer, device)
        message = ", ".join(f"{name}={value:.4f}" for name, value in losses.items())
        writer.add_scalar('epoch loss',losses["total"],epoch+1)
        print(f"epoch {epoch + 1}: {message}")
    save_path = Path.cwd() / "faster_rcnn.pth"
    torch.save(model.state_dict(),save_path)

main

In [9]:
def main() -> None:
    # 1. 基本训练参数
    epochs = 20
    num_classes = 21          # VOC: 20个前景类别 + 1个背景类别

    # 2. 数据集保存位置
    data_root = str(Path.cwd() / "data")

    # 3. 创建 VOC2007 DataLoader
    train_loader = build_voc2007_dataloader(
        root=data_root,
        image_size=(448, 448),
        image_set="trainval",
        train=True,
        download=True,
        num_workers=0,
    )

    # 4. TensorBoard
    writer = SummaryWriter(log_dir=str(Path.cwd() / "logs"))

    # 5. 开始训练
    traintotal(
        epochs=epochs,
        num_classes=num_classes,
        dataloder=train_loader,
        writer=writer,
    )

    writer.close()
if __name__ == "__main__":
    main()

KeyboardInterrupt: 

test

In [10]:
#输出经fliter和nms的预测框，labels，scores
@torch.no_grad()
def predict_one_image(
    model,
    image,
    device,
    score_thresh=0.3,
    nms_thresh=0.3,
    max_detections=100,
):
    """
    image: Tensor[3, H, W]，已经完成和训练时相同的归一化
    """

    model.eval()

    image = image.to(device)
    image_size = (image.shape[-2], image.shape[-1])

    # 模型前向,增加第0维（B）
    output = model(image.unsqueeze(0))

    # 因为一次只预测一张图片
    #proposal是一个list，取出第0维
    proposals = output["proposals"][0]             # [R, 4]
    class_logits = output["roi_class_logits"]      # [R, 21]
    roi_box_deltas = output["roi_box_deltas"]      # [R, 21, 4]

    # 每个 RoI 属于各类别的概率
    class_probabilities = F.softmax(
        class_logits,
        dim=1,
    )

    final_boxes = []
    final_labels = []
    final_scores = []

    # class_id=0 是背景，不参与最终预测
    for class_id in range(1, class_probabilities.shape[1]):

        class_scores = class_probabilities[:, class_id]

        # 第一步：分数过滤
        score_keep = class_scores >= score_thresh

        if not score_keep.any():
            continue

        #fliter score低的框
        selected_scores = class_scores[score_keep]
        selected_proposals = proposals[score_keep]

        # 保留当前类别对应的框回归结果
        selected_deltas = roi_box_deltas[
            score_keep,
            class_id,
        ]

        # proposal + RoI 偏移量 -> 最终预测框
        predicted_boxes = decode_boxes(
            selected_deltas,
            selected_proposals,
        )

        predicted_boxes = clip_boxes(
            predicted_boxes,
            image_size,
        )

        # 过滤无效框
        widths = predicted_boxes[:, 2] - predicted_boxes[:, 0]
        heights = predicted_boxes[:, 3] - predicted_boxes[:, 1]

        valid = (
            (widths > 1)
            & (heights > 1)
            & torch.isfinite(predicted_boxes).all(dim=1)
            & torch.isfinite(selected_scores)
        )

        predicted_boxes = predicted_boxes[valid]
        selected_scores = selected_scores[valid]

        if predicted_boxes.shape[0] == 0:
            continue

        # 第二步：每个类别单独进行 NMS
        keep = nms(
            predicted_boxes,
            selected_scores,
            nms_thresh,
        )

        predicted_boxes = predicted_boxes[keep]
        selected_scores = selected_scores[keep]

        final_boxes.append(predicted_boxes)
        final_scores.append(selected_scores)

        final_labels.append(
            torch.full(
                (len(keep),),
                class_id,
                dtype=torch.long,
                device=device,
            )
        )

    # 如果没有检测结果
    if not final_boxes:
        return {
            "boxes": torch.empty((0, 4)),
            "labels": torch.empty((0,), dtype=torch.long),
            "scores": torch.empty((0,)),
        }

    final_boxes = torch.cat(final_boxes)
    final_labels = torch.cat(final_labels)
    final_scores = torch.cat(final_scores)

    # 所有类别合并后，保留分数最高的前 max_detections 个
    order = final_scores.argsort(descending=True)
    order = order[:max_detections]

    return {
        "boxes": final_boxes[order].cpu(),
        "labels": final_labels[order].cpu(),
        "scores": final_scores[order].cpu(),
    }
#绘图函数
def show_prediction(image, prediction, title="Faster R-CNN prediction"):
    """
    将归一化图像恢复后显示，并绘制预测框。
    """

    mean = torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(3, 1, 1)

    # 反归一化
    display_image = image.cpu() * std + mean
    display_image = display_image.clamp(0, 1)
    display_image = display_image.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(display_image)

    boxes = prediction["boxes"]
    labels = prediction["labels"]
    scores = prediction["scores"]

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()

        rectangle = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="red",
            linewidth=2,
        )
        ax.add_patch(rectangle)

        class_name = VOC_CLASSES[label.item() - 1]

        ax.text(
            x1,
            y1,
            f"{class_name}: {score.item():.2f}",
            color="white",
            fontsize=10,
            bbox={
                "facecolor": "red",
                "alpha": 0.7,
                "pad": 2,
            },
        )

    ax.set_title(
        f"{title} | detections={len(boxes)}"
    )
    ax.axis("off")
    plt.show()
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

data_root = str(Path.cwd() / "data")

# 验证集不使用随机水平翻转
val_dataset = VOCDataset(
    root=data_root,
    image_set="test",
    image_size=(448, 448),
    train=False,
    download=True,
)

# 加载训练好的模型
model = FasterRCNN(num_classes=21).to(device)

checkpoint_path = Path.cwd() / "faster_rcnn.pth"

state_dict = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(state_dict)
model.eval()

# 随机抽取一张验证图片
random_index = torch.randint(
    low=0,
    high=len(val_dataset),
    size=(1,),
).item()

image, gt_boxes, gt_labels = val_dataset[random_index]

prediction = predict_one_image(
    model=model,
    image=image,
    device=device,
    score_thresh=0.3,
    nms_thresh=0.3,
    max_detections=50,
)

show_prediction(
    image,
    prediction,
    title=f"VOC2007 val index={random_index}",
)

100%|██████████| 451M/451M [01:10<00:00, 6.44MB/s] 


FileNotFoundError: [Errno 2] No such file or directory: 'd:\\miniconda\\internshipproject\\week5\\Faster R-CNN\\faster_rcnn.pth'